In [1]:
SYSTEM_PROMPT = """
You are an expert data annotator preparing a dataset for an instruction-tuning pipeline for a Qwen model.
I will provide you with a piece of polished social media product marketing copywriting. Your job is to reverse-engineer this text and extract the minimal, basic user inputs that a non-technical user would have provided to generate it.

Analyze the text and output a JSON object containing EXACTLY these three keys every single time:
- "company_name": Identify the company or brand name. If none is explicitly mentioned, output "Unknown".
- "product_name": Identify the main product or service being sold. Keep it simple and use everyday language.
- "location": Identify the location (e.g., city, neighborhood, or store branch) ONLY if explicitly mentioned in the text. If no location is stated, you must output null.
- "price": Extract the exact price ONLY if it is explicitly mentioned in the text. If no price is stated, you must output null.
- "benefit": Extract the primary selling point, feature, or promotion (e.g., "tahan lama", "porsi besar", "gratis ongkir", "diskon 50%", "promo buy 1 get 1") ONLY if explicitly mentioned. Keep it to a few simple words. If no benefit or promo is stated, you must output null.

CRITICAL RULES:
1. Always output exactly three keys. Never add extra keys, and never omit a key.
2. Keep extracted values as simple, everyday words. Simulate a low-technical user.
3. Do not invent, estimate, or guess prices. 
4. Output strictly valid JSON only. Do not include Markdown formatting, code blocks (```json), or any conversational filler.
"""

TEST = {
  "instruction": "Write a social media product marketing post.",
  "input": "Company: TechGear\nProduct: SoundMax Pro Headphones\nMain Benefit: 40 hours battery life\nSpecial Offer: 15% off until Friday",
  "output": "Tired of your music dying mid-commute? 🎧 Say hello to the SoundMax Pro Headphones from TechGear! With a massive 40-hour battery life, you can listen all week on a single charge. 🔋 Grab yours now and get 15% off until Friday! Click the link in our bio to upgrade your audio game. 👇✨ #TechGear #AudioUpgrade"
}

In [2]:
from google import genai
from dotenv import load_dotenv, find_dotenv

In [3]:
load_dotenv(find_dotenv())

True

In [4]:
model="gemini-2.5-flash-lite"

In [5]:
client = genai.Client()
response = client.models.generate_content(
    model=model, 
    contents="Say hello"
)
print(response.text)

Hello! How can I help you today?


In [6]:
import json
import time
import pandas as pd

In [7]:
df = pd.read_csv("../../BACKUP/data_4.csv")
df.head()

,video_id,url,description,hashtags
0,7625640084372540693,https://www.tiktok.com/@aerostreet/video/76256...,‼️ SECRET RELEASE RYOMEN SUKUNA ‼️ GRATIS untu...,"#aerostreet, #aerostreetjujutsukaisen, #lokalt..."
1,7625604635604126996,https://www.tiktok.com/@aerostreet/video/76256...,‼️ H-1 SECRET RELEASE RYOMEN SUKUNA ‼️ GRATIS ...,"#aerostreet, #aerostreetjujutsukaisen, #lokalt..."
2,7625533555464015125,https://www.tiktok.com/@aerostreet/video/76255...,‼️ SECRET RELEASE RYOMEN SUKUNA ‼️ GRATIS untu...,"#aerostreet, #aerostreetjujutsukaisen, #lokalt..."
3,7625488314488933653,https://www.tiktok.com/@aerostreet/video/76254...,‼️ SECRET RELEASE RYOMEN SUKUNA ‼️ GRATIS untu...,"#aerostreet, #aerostreetjujutsukaisen, #lokalt..."
4,7625248493120015637,https://www.tiktok.com/@aerostreet/video/76252...,H-2 RELEASE ONLINE Aerostreet | Jujutsu Kaisen...,"#aerostreet, #aerostreetjujutsukaisen, #lokalt..."


In [8]:
df = df["description"]

In [9]:
user_prompt = df[0]
response = client.models.generate_content(
  model=model,
  config=genai.types.GenerateContentConfig(
    system_instruction=SYSTEM_PROMPT,
    response_mime_type="application/json",
    temperature=0.9
  ),
  contents=user_prompt
)
raw = response.text
batch = json.loads(raw)

In [10]:
clean_string = ", ".join(f"{k}: {v}" for k, v in batch.items())
print(clean_string)

company_name: Aerostreet, product_name: Ryomen Sukuna shoes, location: None, price: None, benefit: Gratis untuk 20 orang pemenang


In [11]:
from google.genai import errors
import os
import sys
import time
import json
from pathlib import Path


In [12]:

# Make sure your google genai imports are present here

MAX_RETRIES = 5
try:
    SCRIPT_DIR = Path(__file__).resolve().parent
except NameError:
    SCRIPT_DIR = Path.cwd()

# FIX 1 & 3: Correct pathing, ensure directory exists, and use .jsonl
checkpoint_file = SCRIPT_DIR / "datasets" / "labelled_data.jsonl"
checkpoint_file.parent.mkdir(parents=True, exist_ok=True) 

if not os.path.exists(checkpoint_file):
    with open(checkpoint_file, "w", encoding="utf-8") as f:
        pass

# Note: Change range(1) to range(len(df)) when ready for full run
for i in range(len(df)): 
    user_prompt = df[i]
    retries = 0
    
    while retries < MAX_RETRIES:
        try:
            response = client.models.generate_content(
                model=model,
                config=genai.types.GenerateContentConfig(
                    system_instruction=SYSTEM_PROMPT,
                    response_mime_type="application/json",
                    temperature=0.9
                ),
                contents=user_prompt
            )
            break
        except errors.APIError as e:
            if e.code == 429:
                retries += 1
                wait = 25 * retries  # incremental backoff
                print(f"# 429 - rate limited, retry {retries}/{MAX_RETRIES} in {wait}s")
                time.sleep(wait)
            elif e.code == 503:
                retries += 1
                print(f"# 503 - model overloaded, retry {retries}/{MAX_RETRIES}")
                time.sleep(40)
            else:                    
                print(f"# Unhandled API Error: {e.code} - {e.message}")    
                raise e
    else:
        print(f"# Max retries hit at index {i}, aborting.")
        sys.exit(1)
    
    try:
        if not response.text:
            print(f"# Empty response at index {i} (likely safety filter) — skipping")
            continue
        batch = json.loads(response.text)
    except (json.JSONDecodeError, ValueError) as e:
        print(f"# Parse/Value error at index {i}: {e} — skipping")
        time.sleep(15)
        continue
    
    # FIX 2: Filter out nulls (None) and "Unknown" so they don't appear in the prompt
    clean_parts = []
    for k, v in batch.items():
        if v is not None and v != "Unknown":
            clean_parts.append(f"{k}: {v}")
    
    clean_string = ", ".join(clean_parts)
    
    data = {
        "instruction": "You are a friendly, expert copywriter. Write clear, engaging marketing copy based on the simple brief provided.",
        "input": clean_string, 
        "output": df[i]
    }

    with open(checkpoint_file, "a", encoding="utf-8") as f:
        f.write(json.dumps(data, ensure_ascii=False) + "\n")
        
    print(f"Successfully processed index {i}")
    time.sleep(35) # Your existing rate-limit padding

Successfully processed index 0
Successfully processed index 1
Successfully processed index 2
Successfully processed index 3
Successfully processed index 4
Successfully processed index 5
# 429 - rate limited, retry 1/5 in 25s
# 429 - rate limited, retry 2/5 in 50s
# 429 - rate limited, retry 3/5 in 75s
# 429 - rate limited, retry 4/5 in 100s
# 429 - rate limited, retry 5/5 in 125s
# Max retries hit at index 6, aborting.


SystemExit: 1

c:\Users\lenovo\Documents\GitHub\AI_TAHAP_2\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3755: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
